## Analyze results good-quality mapping

In [ ]:
import scanpy as sc
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../../../")
sys.path.append("../../utils/")
import mapqc
import data_cleanup
import analysis

In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

params

In [ ]:
study = "Sheppard_2020"
batch_variable = "sample"
testrun = False
use_old_counts = True
kl = 1.0

paths:

In [ ]:
if testrun:
    suffix = "_test"
    suffix_2 = ""
else:
    suffix = ""
    if use_old_counts == True:
        suffix_2 = "original_counts_"
    else:
        suffix_2 = "cellxgene_counts_"

path_emb = f"/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_{study}_{suffix_2}batch_{batch_variable}_kl{kl}_latent{suffix}.h5ad"
path_counts = f"/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_{study}_counts{suffix}.h5ad"
path_mapqc_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/atlas_extension/results/HLCA/{study}/{suffix_2}batch_{batch_variable}_kl{kl}{suffix}/mapqc"

load relevant files:

In [ ]:
path_counts

In [ ]:
path_emb

In [ ]:
adata_emb = sc.read_h5ad(path_emb)
adata_counts = sc.read_h5ad(path_counts)
adata = data_cleanup.add_mapping_info_to_counts_adata(adata_counts, adata_emb)

set gene symbols as adata.var row names (instead of IDs):

In [ ]:
adata.var["ensembl_id"] = adata.var.index
adata.var.index = [
    fn if not pd.isnull(fn) else gid
    for fn, gid in zip(adata.var.feature_name, adata.var.ensembl_id)
]

In [ ]:
adata.X[-6:, -6:].toarray()

In [ ]:
results_healthy = mapqc.load(os.path.join(path_mapqc_results_dir, "control.pkl"))
results_disease = mapqc.load(os.path.join(path_mapqc_results_dir, "case.pkl"))

copy results to main adata:

In [ ]:
for stat in [
    "mapqc_score",
    "mapqc_score>2",
]:
    adata.obs[stat] = np.nan
    healthy_idc = [
        idx for idx in adata.obs.index if idx in results_healthy.adata.obs.index
    ]
    adata.obs.loc[healthy_idc, stat] = results_healthy.adata.obs.loc[healthy_idc, stat]
    disease_idc = [
        idx for idx in adata.obs.index if idx in results_disease.adata.obs.index
    ]
    adata.obs.loc[disease_idc, stat] = results_disease.adata.obs.loc[disease_idc, stat]

correct annotation hierarchy mistake in HLCA: pericytes are said to be part of the fibroblast supergroup, when they are actually a separate cell type

In [ ]:
for lev in [2, 3]:
    for ann_colname in [f"ann_level_{lev}", f"ann_level_{lev}_pred"]:
        adata.obs[ann_colname] = adata.obs[ann_colname].tolist()
        results_healthy.adata.obs[ann_colname] = results_healthy.adata.obs[
            ann_colname
        ].tolist()
        results_disease.adata.obs[ann_colname] = results_disease.adata.obs[
            ann_colname
        ].tolist()
        if "_pred" in ann_colname:
            ref_col = "ann_level_5_pred"
        else:
            ref_col = "ann_level_5"
        adata.obs.loc[
            adata.obs[ref_col] == "Pericytes",
            ann_colname,
        ] = "Pericytes"
        results_healthy.adata.obs.loc[
            results_healthy.adata.obs[ref_col] == "Pericytes",
            ann_colname,
        ] = "Pericytes"
        results_disease.adata.obs.loc[
            results_disease.adata.obs[ref_col] == "Pericytes",
            ann_colname,
        ] = "Pericytes"

## Plots:

In [ ]:
FIGURES = dict()

UMAP colored by reference, query control, query case

In [ ]:
# create variable splitting data in ref control, query control
adata.obs["r_or_q_ctr_or_case"] = [
    f"{r_q} {stat}" for r_q, stat in zip(adata.obs.r_or_q, adata.obs.status)
]
# clean up:
cleaning_dict = {
    "r control": "Reference (controls)",
    "q control": "Query controls",
    "q IPF": "Query IPF",
}
adata.obs.r_or_q_ctr_or_case = adata.obs.r_or_q_ctr_or_case.map(cleaning_dict)
results_healthy.adata.obs["r_or_q_ctr_or_case"] = adata.obs.loc[
    results_healthy.adata.obs.index, "r_or_q_ctr_or_case"
].copy()
results_disease.adata.obs["r_or_q_ctr_or_case"] = adata.obs.loc[
    results_disease.adata.obs.index, "r_or_q_ctr_or_case"
].copy()
palette_1 = {
    "Reference (controls)": "grey",
    "Query controls": "dodgerblue",
    "Query IPF": "midnightblue",
}

In [ ]:
sc.set_figure_params(figsize=(6, 6))

FIGURES["umap_r_or_q_ctr_or_case"] = sc.pl.umap(
    adata,
    color="r_or_q_ctr_or_case",
    frameon=False,
    ncols=1,
    palette=palette_1,
    show=False,
    s=0.7,
)

UMAP colored by cell type annotation:  

In [ ]:
for lev in range(1, 6):
    adata.obs[f"ann_level_{lev}_merged"] = [
        ann if not pd.isnull(ann) else ann_pred
        for ann, ann_pred in zip(
            adata.obs[f"ann_level_{lev}"], adata.obs[f"ann_level_{lev}_pred"]
        )
    ]

In [ ]:
FIGURES["umap_anns"], ax = plt.subplots()
sc.pl.umap(adata, color="ann_level_3_merged", frameon=False, ax=ax, s=0.7)

UMAP colored by mapqc score controls:

In [ ]:
FIGURES["umap_mapqc_score"] = mapqc.pl.umap_mapqc_scores(
    results_healthy, return_fig=True
)

In [ ]:
FIGURES["umap_ctr_and_case_mapqc_score_binarized"], ax = plt.subplots(figsize=(6, 6))
sc.pl.umap(
    adata,
    color="mapqc_score>2",
    # return_fig=True,
    size=1,
    palette={
        ">2": "firebrick",
        "<=2": "navajowhite",
        "filtered out": "black",
        "not sampled": "cornflowerblue",
    },
    frameon=False,
    na_color="gray",
    ax=ax,
)

In [ ]:
FIGURES["umap_ctr_mapqc_score_binarized"] = mapqc.pl.umap_mapqc_scores(
    results_healthy, color_by="mapqc_score>2", return_fig=True, size=0.5
)

In [ ]:
FIGURES["umap_case_mapqc_score_binarized"] = mapqc.pl.umap_mapqc_scores(
    results_disease, color_by="mapqc_score>2", return_fig=True
)

In [ ]:
mapqc.pl.umap_filtering_info(results_disease)

In [ ]:
sc.pl.umap(adata, color=["CTHRC1", "study", "ann_level_4_merged"])

Boxplot showing mapqc scores in controls per cell type:

In [ ]:
FIGURES["mapqc_per_ct_ctr_only"] = mapqc.pl.boxplot_mapqc_scores_per_category(
    results_healthy,
    box_category="ann_level_2_pred",
    figsize=(5, 4.5),
    point_size=0.3,
    color_category="donor_id",
    return_fig=True,
    min_cells_per_cat=150,
)

In [ ]:
FIGURES[
    "grouped_boxplot_lev2_ann_mapqc_score"
] = mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_2_pred",
    group_category="r_or_q_ctr_or_case",
    palette=palette_1,
    figsize=(5, 4),
    vmax=15,
    min_cells_per_cat=100,
    return_fig=True,
)
plt.xlabel("Cell type")
plt.show()

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_3_pred",
    group_category="r_or_q_ctr_or_case",
    palette=palette_1,
    figsize=(12, 5),
    vmax=15,
    min_cells_per_cat=50,
)

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_4_pred",
    group_category="r_or_q_ctr_or_case",
    palette=palette_1,
    figsize=(12, 5),
    vmax=15,
    min_cells_per_cat=50,
)

### Analysis endothelial cells:

In [ ]:
FIGURES[
    "grouped_barplot_end_subtype_mapqc_scores"
] = mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_5_pred",
    group_category="r_or_q_ctr_or_case",
    box_groups_to_include=[
        "EC aerocyte capillary",
        "EC arterial",
        "EC general capillary",
        "EC venous pulmonary",
        "EC venous systemic",
    ],
    figsize=(5, 5),
    palette=palette_1,
    min_cells_per_cat=50,
    return_fig=True,
)
plt.xlabel("Cell type")
plt.show()

subset to endothelial cells:

In [ ]:
adata_end = adata[adata.obs.ann_level_2_merged == "Blood vessels", :].copy()

re-run knn graph and umap

In [ ]:
sc.pp.neighbors(adata_end, n_neighbors=30, use_rep="X_scanvi_emb")
sc.tl.umap(adata_end)

In [ ]:
sc.tl.leiden(adata_end, key_added="leiden_end", resolution=0.3)

clean up EC annotations:

In [ ]:
adata_end.obs["ann_level_5_cleaned"] = adata_end.obs.ann_level_5_merged.copy()
end_ct_count = adata_end.obs.ann_level_5_cleaned.value_counts()
min_n_cells = 50
low_n_anns = end_ct_count.index[end_ct_count < min_n_cells]
renamer = {
    (ct): ("Other" if ct in low_n_anns else ct)
    for ct in adata_end.obs.ann_level_5_cleaned.unique()
}
adata_end.obs.ann_level_5_cleaned = adata_end.obs.ann_level_5_cleaned.map(renamer)
# palette_2 = {}
# cmap = plt.get_cmap("tab10").colors
# num_colors = len(cmap)

# for index, category in enumerate(adata_end.obs.ann_level_5_cleaned.unique()):
#     color = cmap[index % num_colors]
#     palette_2[category] = color
# palette_2["Other"] = "tab:grey"
palette_2 = {
    "EC aerocyte capillary": "#377eb8",  # "tab:blue",
    "EC arterial": "#ff7f00",  # "tab:orange",
    "EC general capillary": "#4daf4a",
    "EC venous pulmonary": "#f781bf",
    "EC venous systemic": "#a65628",
    "Other": "#a9a9a9",
}  # '#984ea3','#999999', '#e41a1c', '#dede00']

In [ ]:
with plt.rc_context({"figure.figsize": (2, 2)}):
    sc.pl.umap(
        adata_end, color="ann_level_5_cleaned", palette=palette_2, frameon=False, s=0.5
    )

In [ ]:
sc.pl.umap(
    adata_end,
    color=[
        "r_or_q",
        "mapqc_score>2",
        "lung_condition",
        "leiden_end",
    ],
    wspace=0.3,
    frameon=False,
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    mapqc_results=results_healthy,
    box_category="ann_level_5_pred",
    groups_to_include=[
        "EC aerocyte capillary",
        "EC arterial",
        "EC general capillary",
        "EC venous pulmonary",
        "EC venous systemic",
    ],
    figsize=(2, 4),
    color_high_median_boxes_red=True,
    min_cells_per_cat=50,
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    mapqc_results=results_disease,
    box_category="ann_level_5_pred",
    groups_to_include=[
        "EC aerocyte capillary",
        "EC arterial",
        "EC general capillary",
        "EC venous pulmonary",
        "EC venous systemic",
    ],
    figsize=(2, 4),
    min_cells_per_cat=50,
    color_high_median_boxes_red=True,
)

show cluster make-up:

In [ ]:
FIGURES["EC_cluster_barplot"], ax = plt.subplots(figsize=(4, 2))
pd.crosstab(adata_end.obs.leiden_end, adata_end.obs.r_or_q_ctr_or_case).plot(
    kind="bar", stacked=True, color=palette_1, ax=ax
)
leg = plt.legend(loc=(1.01, 0.4), frameon=False, title="Cell source")
leg._legend_box.align = "left"
ax.spines[["right", "top"]].set_visible(False)
ax.set_ylabel("N cells")
ax.set_xlabel("Endothelial cell cluster")
plt.show()

In [ ]:
FIGURES["EC_subtype_barplot"], ax = plt.subplots(figsize=(4, 2))
pd.crosstab(adata_end.obs.leiden_end, adata_end.obs.ann_level_5_cleaned).plot(
    kind="bar", stacked=True, ax=ax, color=palette_2
)
leg = plt.legend(loc=(1.01, 0), frameon=False, title="EC subtype")
leg._legend_box.align = "left"
ax.spines[["right", "top"]].set_visible(False)
ax.set_ylabel("N cells")
ax.set_xlabel("Endothelial cell cluster")
plt.show()

marker plot:

In [ ]:
EC_markers = {
    "EC": ["SPARCL1", "AQP1", "RNASE1"],
    "EC arterial": ["DKK2", "IGFBP3"],
    "EC aerocyte capillary": ["SOSTDC1"],
    "EC general capillary": ["IL7R", "FCN3", "MT1M"],
    "EC venous systemic": ["TPD52L1", "ZNF385D", "OLFM1"],
    "EC venous pulmonary": [
        "CPE",
        "C7_ENSG00000112936",
        "MMRN1",
    ],  # gene with long name is actually C7 but somehow it is named with ensembl id here...
}

In [ ]:
adata_end.obs["ct_r_or_q"] = [
    f"{ct} ({r_or_q})"
    for ct, r_or_q in zip(adata_end.obs.ann_level_5_cleaned, adata_end.obs.r_or_q)
]

In [ ]:
sc.pl.dotplot(
    adata_end, var_names=EC_markers, groupby="ct_r_or_q", standard_scale="var"
)

### Pericyte/smooth muscle analysis:

subset to mesenchyme except mesothelium:

In [ ]:
adata_mes = adata[
    adata.obs.ann_level_2_merged.isin(
        ["Pericytes", "Smooth muscle", "Fibroblast lineage"]
    ),
    :,
].copy()

In [ ]:
del adata_mes.uns["ann_level_3_merged_colors"]

In [ ]:
sc.pp.filter_genes(adata_mes, min_cells=5)

In [ ]:
adata.shape

In [ ]:
adata_mes.shape

In [ ]:
sc.pp.neighbors(adata_mes, n_neighbors=30, use_rep="X_scanvi_emb")
sc.tl.umap(adata_mes)

In [ ]:
sc.tl.leiden(adata_mes, key_added="leiden_mes", resolution=0.3)

In [ ]:
sc.pl.umap(
    adata_mes,
    color=[
        "r_or_q",
        "mapqc_score>2",
        "lung_condition",
        "leiden_mes",
        "study",
        "ann_level_3_merged",
        "ann_level_4_merged",
    ],
    wspace=0.3,
    frameon=False,
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    mapqc_results=results_disease,
    box_category="ann_level_3_pred",
    color_category="sample",
    groups_to_include=["Smooth muscle", "Pericytes"],
    figsize=(4, 4),
    min_cells_per_cat=50,
    color_high_median_boxes_red=False,
)

##### gene-level analysis:

create obs column combining mapqc binary score and cell type annotation, both for adata_mes and for adata

In [ ]:
# adata_mes (use level 3)
for lev in [3, 4]:
    obs_col = f"ann_level_{lev}_mapqc_score"
    adata_mes.obs[obs_col] = [
        f"{ct} ({stat})"
        for ct, stat in zip(
            adata_mes.obs[f"ann_level_{lev}_merged"],
            adata_mes.obs["mapqc_score>2"],  # .r_or_q_ctr_or_case
        )
    ]
    if lev == 3:
        min_n_cells = 200
    if lev == 4:
        min_n_cells = 50
    adata_mes.obs[obs_col] = analysis.clean_up_df_column(
        adata_mes.obs, column_name=obs_col, min_n_cells_per_cat=min_n_cells
    )
    adata_mes.obs[obs_col] = adata_mes.obs[obs_col].str.replace("nan", "ref.")
    adata_mes.obs[obs_col] = adata_mes.obs[obs_col].str.replace("<=2", "q., mapqc<2")
    adata_mes.obs[obs_col] = adata_mes.obs[obs_col].str.replace(">2", "q., mapqc>2")
# adata (use level 2)
for lev in [2, 3, 4]:
    obs_col = f"ann_level_{lev}_mapqc_score"
    adata.obs[obs_col] = [
        f"{ct} ({stat})"
        for ct, stat in zip(
            adata.obs[f"ann_level_{lev}_merged"],
            adata.obs["mapqc_score>2"],
        )
    ]
    if lev in [2, 3]:
        min_n_cells = 200
    if lev == 4:
        min_n_cells = 50
    adata.obs[obs_col] = analysis.clean_up_df_column(
        adata.obs, column_name=obs_col, min_n_cells_per_cat=min_n_cells
    )
    adata.obs[obs_col] = adata.obs[obs_col].str.replace("nan", "ref.")
    adata.obs[obs_col] = adata.obs[obs_col].str.replace("<=2", "q., mapqc<2")
    adata.obs[obs_col] = adata.obs[obs_col].str.replace(">2", "q., mapqc>2")

DEA on pericytes >2:

In [ ]:
sc.tl.rank_genes_groups(
    adata_mes,
    groupby="ann_level_3_mapqc_score",
    groups=["Pericytes (q., mapqc>2)"],
    reference="Pericytes (ref.)",
    key_added="Pericytes (>2)",
)

In [ ]:
sc.tl.filter_rank_genes_groups(
    adata_mes, key="Pericytes (>2)", key_added="Pericytes (>2) filtered"
)  # , max_out_group_fraction=0.1)
peri_bad_map_deg_df = sc.get.rank_genes_groups_df(
    adata_mes, group="Pericytes (q., mapqc>2)", key="Pericytes (>2) filtered"
).dropna(how="any")
peri_bad_map_deg_df = peri_bad_map_deg_df.loc[
    peri_bad_map_deg_df.pvals_adj < 0.05, :
]  # only significant
peri_bad_map_deg_df = peri_bad_map_deg_df.loc[
    peri_bad_map_deg_df.scores > 0, :
]  # only upregulated, for downregulated this filtering does not make sense

the resulting genes are often genes also highly expressed in SM. Now also do strict filtering. to find genes unique to the >2 pericyte group:

In [ ]:
sc.tl.filter_rank_genes_groups(
    adata_mes,
    max_out_group_fraction=0.1,
    key="Pericytes (>2)",
    key_added="Pericytes (>2) filtered strict",
)
peri_bad_map_deg_df_strict = sc.get.rank_genes_groups_df(
    adata_mes, group="Pericytes (q., mapqc>2)", key="Pericytes (>2) filtered strict"
).dropna(how="any")
peri_bad_map_deg_df_strict = peri_bad_map_deg_df_strict.loc[
    peri_bad_map_deg_df_strict.pvals_adj < 0.05, :
]  # only significant
peri_bad_map_deg_df_strict = peri_bad_map_deg_df_strict.loc[
    peri_bad_map_deg_df_strict.scores > 0, :
]  # only upregulated, for downregulated this filtering does not make sense

In [ ]:
peri_bad_map_low_deg_df = sc.get.rank_genes_groups_df(
    adata_mes, group="Pericytes (q., mapqc>2)", key="Pericytes (>2)"
).iloc[-30:, :]

DEA on smooth muscle >2

In [ ]:
sc.tl.rank_genes_groups(
    adata_mes,
    groupby="ann_level_3_mapqc_score",
    groups=["Smooth muscle (q., mapqc>2)"],
    reference="Smooth muscle (ref.)",
    key_added="Smooth muscle (>2)",
)

In [ ]:
sc.tl.filter_rank_genes_groups(
    adata_mes,
    max_out_group_fraction=0.1,
    key="Smooth muscle (>2)",
    key_added="Smooth muscle (>2) filtered strict",
)
sm_bad_map_deg_df_strict = sc.get.rank_genes_groups_df(
    adata_mes,
    group="Smooth muscle (q., mapqc>2)",
    key="Smooth muscle (>2) filtered strict",
).dropna(how="any")
sm_bad_map_deg_df_strict = sm_bad_map_deg_df_strict.loc[
    sm_bad_map_deg_df_strict.pvals_adj < 0.05, :
]  # only significant
sm_bad_map_deg_df_strict = sm_bad_map_deg_df_strict.loc[
    sm_bad_map_deg_df_strict.scores > 0, :
]  # only upregulated, for downregulated this filtering does not make sense

plot downregulated genes:

In [ ]:
sc.pl.dotplot(
    adata_mes,
    var_names=peri_bad_map_low_deg_df.names,
    groupby="ann_level_3_mapqc_score",
    standard_scale="var",
)

plot upregulated genes:

In [ ]:
sc.pl.dotplot(
    adata_mes,
    var_names=peri_bad_map_deg_df.names.tolist(),
    groupby="ann_level_3_mapqc_score",
    dot_max=1,
    standard_scale="var",
)

and upregulated genes after strict filtering:

In [ ]:
sc.pl.dotplot(
    adata_mes,
    var_names=peri_bad_map_deg_df_strict.names.tolist(),
    groupby="ann_level_3_mapqc_score",
    dot_max=1,
    standard_scale="var",
)

ARHGAP15: tumor suppressor in several cancers (stops cell growth I guess)
HOPX: same. "MAP3K7CL": pericyte subtypes in mouse *embryo* (https://www.nature.com/articles/s41467-018-07770-1#MOESM1), see also: https://cellxgene.cziscience.com/collections/48d354f5-a5ca-4f35-a3bb-fa3687502252. Seems like this is related to foetal stage pericyte? 
!!! See also here: https://cellxgene.cziscience.com/e/3dc61ca1-ce40-46b6-8337-f27260fd9a03.cxg/ this is human! Vascular SMC2, just like in mouse paper :) From this paper, difference between two "vascular SMC" groups. Group 1: ["GPC3","SPON2","TGFB1"]. Group 2: ["NET1","PLN","ADIRF"]

check results in whole adata:

In [ ]:
markers_mes = {
    "pericyte bad mapping": peri_bad_map_deg_df.names.tolist(),
    "pericyte bad mapping strict filt": peri_bad_map_deg_df_strict.names.tolist(),
    "pericyte": ["LAMC3", "HIGD1B", "PTN"],
    "smooth muscle": [
        "MYH11",
        "TINAGL1",
        "PLN",
        "ACTA2",
        "TAGLN",
    ],  # last two from Martijn and Elo, first three from HLCA
    "smooth mucle bad mapping": sm_bad_map_deg_df_strict.names.tolist(),
    "AT0 or alveolar epithelium": [
        "SLC39A8",  # from here on: AT0 and alveolar epithelium markers
        "SPOCK2",
        "MFSD2A",
        "BIRC5",
        "SFTPB",
    ],
    "fetal pericyte paper 1 and DEG": ["MAP3K7CL"],
    "fetal pericyte 1": ["GPC3", "SPON2", "TGFB1"],
    "fetal pericyte 2": [
        "NET1",
        "PLN",
        "ADIRF",
        "FRZB",
        "NUDT4",
    ],  # a few of these are re-expressed in IPF-specific pericytes!
}  # check other alveolar epithelium markers to check for doublets or misannotations

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=markers_mes,
    groupby="ann_level_2_mapqc_score",
    standard_scale="var",
)

Conclusion: there is some pericyte-smooth muscle intermediate state, with cells expressing both a subset of pericyte markers (HIGD1B, PTN, rarely expressed in SM), and genes normally only (in the stroma) expressed in SM (RGS16, FRZB, MAP3K7CL, STEAP4 and more, these are outcome of DEA of Pericytes mapqc>2 versus pericyes ref, with ref being consistent with <2 query). Does this group of cells also show a unique marker --> ARHGAP15

To show: whereas IPF pericytes show normal expression of some SM markers (PLN is low) and of some pericyte markers (HIG1DB high), they show increased expression of other SM markers or SM specific genes (RGS16, FRZB, MAP3K7CL, STEAP4), and decreased expression of some pericyte markers (LAMC3 low).<br>
Similarly, whereas IPF smooth muscle show normal or higher expression of SM markers (RGS16, FRZB, STEAP4, MAP3K7CL extra high, PLN a little bit lower), they show increased expression of pericyte markers (mostly PTN).<br>
Check 1: is this due to mis-annotations? Then there would be no or few double-positives, but in fact the number of DPs is very high in pericytes (see e.g. HIG1DB-RGS16 or HIG1DB-FRZB), and also relatively high in SM (see PTN_PLN or PTN_FRZB).<br>
Check 2: are these not doublets? This can only be proven by gene more highly expressed in this double-positive group specifically than in any of the other two groups. This is the case for ARHGAP15, LZTS1, CTSC, HOPX, for the pericyte double positives, and increased expression compared to both SM and pericytes for MAP3K7CL (output of DEA strict filtering for both SM and pericyte).

In [ ]:
groups_of_interest = [
    g
    for g in adata_mes.obs.ann_level_3_mapqc_score.unique()
    if (g.startswith("Peri") or g.startswith("Smooth"))
]

In [ ]:
double_positive_df = pd.DataFrame(index=sorted(groups_of_interest))
single_genes_not_for_dp = ["ARHGAP15", "LZTS1", "CTSC", "HOPX", "MAP3K7CL"]
single_pos_calculated = []
for peri_gene in ["HIGD1B", "PTN"]:
    for sm_gene in ["RGS16", "FRZB", "MAP3K7CL", "STEAP4", "PLN"]:
        gene_df = pd.DataFrame(index=adata_mes.obs.index)
        gene_df["ann"] = adata_mes.obs.ann_level_3_mapqc_score
        gene_df[peri_gene] = adata_mes[:, peri_gene].X.toarray()
        gene_df[sm_gene] = adata_mes[:, sm_gene].X.toarray()
        for single_gene in [peri_gene, sm_gene] + single_genes_not_for_dp:
            if single_gene not in single_pos_calculated:
                if single_gene not in gene_df.columns:
                    gene_df[single_gene] = adata_mes[:, single_gene].X.toarray()
                # calculate positive fraction for single gene:
                gene_df["sp"] = gene_df[single_gene] > 0
                sp_frac = gene_df.groupby("ann")["sp"].apply(lambda x: (x > 0).mean())
                double_positive_df[single_gene] = round(
                    sp_frac.loc[double_positive_df.index], 3
                )
                single_pos_calculated.append(single_gene)
        # calculate positive fraction for single sm gene
        gene_df["dp"] = (gene_df[peri_gene] > 0) & (gene_df[sm_gene] > 0)
        adata_mes.obs[f"dp_{peri_gene}_{sm_gene}"] = gene_df["dp"]
        dp_frac = gene_df.groupby("ann")["dp"].apply(lambda x: (x > 0).mean())
        double_positive_df[f"{peri_gene}_{sm_gene}"] = round(
            dp_frac.loc[double_positive_df.index], 3
        )

In [ ]:
# also calculate mean total counts per group:
adata_mes.obs.total_counts = adata_mes.X.sum(axis=1)
group_mean_total_count = adata_mes.obs.groupby("ann_level_3_mapqc_score").agg(
    {"total_counts": "mean"}
)
double_positive_df["total_count_norm"] = (
    group_mean_total_count.loc[double_positive_df.index] / group_mean_total_count.max()
)

In [ ]:
import seaborn as sns

In [ ]:
sns.heatmap(
    double_positive_df.loc[
        [
            "Pericytes (q., mapqc<2)",
            "Pericytes (q., mapqc>2)",
            "Pericytes (ref.)",
            "Smooth muscle (q., mapqc<2)",
            "Smooth muscle (q., mapqc>2)",
            "Smooth muscle (ref.)",
        ],
        ["total_count_norm"]
        # + sorted(
        #     [
        #         col
        #         for col in double_positive_df.columns
        #         if ("_" in col and col != "total_count_norm")
        #     ]
        + [
            "PTN_FRZB",
            "PTN_MAP3K7CL",
        ]
        + single_genes_not_for_dp,
    ],
    cmap="Reds",
)

check results in umap of mesenchyme:

In [ ]:
for ct in [
    "smooth muscle",
    "pericyte",
    "fetal pericyte paper 1 and DEG",
    "fetal pericyte 2",
]:
    print(ct.upper())
    sc.pl.umap(adata_mes, color=markers_mes[ct], frameon=False, ncols=5)
    print("\n\n\n")
# add handpicked genes for >2 pericytes:
sc.pl.umap(adata_mes, color=markers_mes[ct], frameon=False, ncols=5)

In [ ]:
sc.pl.umap(
    adata_mes,
    color=[
        "r_or_q",
        "mapqc_score>2",
        "lung_condition",
        "leiden_mes",
        "study",
        "ann_level_3_merged",
        "ann_level_4_merged",
    ]
    + [col for col in adata_mes.obs.columns if col.startswith("dp")],
    frameon=False,
    ncols=5,
)

In [ ]:
mes_markers_intermediate = {
    "SM marker low in all pericyte": ["PLN"],
    "Pericyte marker high in all pericyte": ["HIGD1B", "PTN"],  # I need more of these!
    "SM marker high in IPF pericyte": ["RGS16", "FRZB", "MAP3K7CL", "STEAP4"],
    "Pericyte marker low in IPF pericyte": ["LAMC3"],
    "SM marker high in all SM": ["RGS16", "FRZB", "STEAP4", "MAP3K7CL"],
    # "Pericyte marker higher in IPF SM": ["PTN"],
    "Pericyte marker high in IPF peri only": ["ARHGAP15", "LZTS1", "CTSC", "HOPX"],
    "Marker extra high in IPF peri and SM": ["MAP3K7CL"],
    "Pericyte marker low in IPF peri": [
        "TBX3",
        "CPM",
        "EZR",
        "RGN",
        "PAG1",
        "OTULINL",
        "PAG1",
        "KCNK3",
    ],  # cut this down further
    "Fibroblast IPF marker from paper": ["CTHRC1"],
}

In [ ]:
sc.pl.dotplot(
    adata_mes,
    var_names=mes_markers_intermediate,
    groupby="ann_level_3_mapqc_score",
    standard_scale="var",
)

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=mes_markers_intermediate,
    groupby="ann_level_2_mapqc_score",
    # standard_scale="var",
)

Paper text draft: Gene expression patterns suggest that pericytes with high mapQC scores (>2), as observed in large numbers in samples from IPF patients, in fact constitute an intermediate cell state between healthy pericytes and smooth muscle cells. These pericytes show expression of genes normally expressed in smooth muscle and not in pericytes (FRZB, MAP3K7CL), while also consistently expressing (>80% of cells positive) a pericyte-specific marker not expressed in smooth muscle (PTN) [removed HIG1DB from here as it is less clean]. Notably, joint expression of these genes is rarely observed in healthy lung pericytes or smooth muscle cells [fig. heatmap], but has been observed in a subset of pericytes of the developing lung [vascular SMC2]. While not fully recapitulating transcriptomic patterns of this embryonic counterpart, both cell types also [Removing: These pericytes moreover express RGS16, normally primarily expressed in activated or stressed smooth muscle] lack  expression of genes highly expressed in healthy, adult pericytes (RGN, PAG1, TBX3, LAMC3). [removed EZR as not recapitulated in embryonic ct, and also not as clean in HLCA as the others). [removing this as not recapitulated in embryonic state, need to keep it short: or in both smooth muscle and pericytes (EZR, MYH11)].

In [ ]:
markers_paper_figure = {
    "Pericyte,\nall states": ["PTN", "HIGD1B"],
    "Pericyte,\nlacking in IPF state": ["LAMC3", "RGN", "PAG1", "TBX3"],
    "SM, also in IPF pericytes": ["FRZB", "MAP3K7CL"],
    "SM, not in pericytes": ["PLN"],
}

In [ ]:
groups_to_show_hlca = [
    gr
    for gr in adata_mes.obs.ann_level_3_mapqc_score.unique()
    if (("smooth muscle (" in gr.lower()) or ("peri" in gr.lower()))
]
sc.pl.dotplot(
    adata_mes[adata_mes.obs.ann_level_3_mapqc_score.isin(groups_to_show_hlca), :],
    var_names=markers_paper_figure,
    groupby="ann_level_3_mapqc_score",
    standard_scale="var",
    use_raw=False,
)

In [ ]:
adata_developmental = sc.read_h5ad(
    "/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/developing_lung/he_et_al_fibroblasts_sm.h5ad"
)

In [ ]:
adata_developmental.var["ensembl_id"] = adata_developmental.var.index
adata_developmental.var.index = adata_developmental.var.gene_symbols

In [ ]:
groups_to_show_dev = [
    gr
    for gr in adata_developmental.obs.new_celltype.unique()
    if (("sm" in gr.lower()) or ("peri" in gr.lower()))
]
sc.pl.dotplot(
    adata_developmental[
        adata_developmental.obs.new_celltype.isin(groups_to_show_dev), :
    ],
    var_names=markers_paper_figure,
    groupby="new_celltype",
    standard_scale="var",
    use_raw=False,
)

check markers from madissoon et al spatial lung paper (Outcome: healthy pericytes are all pulmonary pericytes, IPF ones (mapqc>2) are likely also but altered, don't show signs of any alternative peri subtypes.)

In [ ]:
madissoon_markers = {
    "ASM": ["HS6ST3", "GPM6A", "DOCK3", "COL4A6", "HPSE2"],
    "IR-Ven-Peri": ["FGF7"],
    "Peri-pulm": ["KCNK3", "MEST", "LAMC3", "PTN"],
    "Peri-syst": ["FRMD3", "SLC38A11"],
    "SM-pulm": ["TNNT2"],  # or for both SM groups: ['TGM2']
    "SM-Art_syst": ["SORBS2", "SYNM", "TGM2"],  # no unique markers...
}

In [ ]:
sc.pl.dotplot(
    adata_mes,
    var_names=madissoon_markers,
    groupby="ann_level_3_mapqc_score",
    standard_scale="var",
)

### Alveolar fibroblast sanity check:

In [ ]:
sc.tl.rank_genes_groups(
    adata_mes,
    groupby="ann_level_4_mapqc_score",
    groups=["Alveolar fibroblasts (q., mapqc>2)"],
    reference="Alveolar fibroblasts (ref.)",
    key_added="Alveolar fibroblasts (>2)",
)

In [ ]:
sc.tl.filter_rank_genes_groups(
    adata_mes,
    key="Alveolar fibroblasts (>2)",
    key_added="Alveolar fibroblasts (>2) filtered",
)  # , max_out_group_fraction=0.1)
alv_fib_bad_map_deg_df = sc.get.rank_genes_groups_df(
    adata_mes,
    group="Alveolar fibroblasts (q., mapqc>2)",
    key="Alveolar fibroblasts (>2) filtered",
).dropna(how="any")
alv_fib_bad_map_deg_df = alv_fib_bad_map_deg_df.loc[
    alv_fib_bad_map_deg_df.pvals_adj < 0.05, :
]  # only significant
alv_fib_bad_map_deg_df = alv_fib_bad_map_deg_df.loc[
    alv_fib_bad_map_deg_df.scores > 0, :
]  # only upregulated, for downregulated this filtering does not make sense

In [ ]:
alv_fib_bad_map_deg_df

In [ ]:
sc.pl.dotplot(
    adata_mes,
    var_names=alv_fib_bad_map_deg_df.names.tolist() + ["CTHRC1"],
    groupby="ann_level_4_mapqc_score",
    standard_scale="var",
)

In [ ]:
sc.pl.umap(
    adata_mes,
    color=["study", "r_or_q", "ann_level_4_pred", "ann_level_4_mapqc_score", "CTHRC1"],
    wspace=0.7,
    ncols=2,
)

plot umap subset of whole data, only fibroblasts:

In [ ]:
sc.pl.umap(
    adata,  # [adata.obs.ann_level_2_merged == "Fibroblast lineage", :],
    color=["ann_level_4", "ann_level_4_merged", "ann_level_4_pred"],
    groups=[
        "Adventitial fibroblasts",
        "Alveolar fibroblasts",
        "Myofibroblasts",
        "Peribronchial fibroblasts",
        "Subpleural fibroblasts",
    ],
    ncols=1,
    frameon=False,
)

In [ ]:
mapqc.pl.boxplot_sample_scores_per_nhood(
    results_disease, label_x_ticks_by="ann_level_4_pred"
)

In [ ]:
sc.pl.umap(adata, color=["CTHRC1"], frameon=False)

In [ ]:
nhood_oi = results_disease.nhood_stats.index[
    results_disease.nhood_stats.nhood_number == 216  # 154
][0]

In [ ]:
mapqc.pl.umap_center_cell(results_disease, neighborhood_center_cell=nhood_oi)

In [ ]:
mapqc.pl.umap_show_neighborhood(results_disease, neighborhood=nhood_oi)

In [ ]:
mapqc.pl.boxplot_sample_scores_per_nhood(
    results_disease,
    neighborhood=nhood_oi,
    use_zscored_dists=True,
    palette={
        "r": "grey",
        "q": "midnightblue",
    },
)

In [ ]:
nhood_adata_to_plot = mapqc.pl.umap_neighborhood(
    results_disease,
    color_by=[
        "sample",
        "study",
        "r_or_q",
        "mapqc_score",
        "mapqc_score>2",
        "ann_level_4_pred",
        "ann_level_4",
    ],
    neighborhood=nhood_oi,
    size=20,
    recalculate_umap=True,
    umap_kwargs={"cmap": "coolwarm", "vmin": -4, "vmax": 4},
    return_adata=True,
)

In [ ]:
nhood_adata_to_plot

In [ ]:
nhood_adata_to_plot.obs["CTHRC1"] = adata[
    nhood_adata_to_plot.obs.index, "CTHRC1"
].X.toarray()

In [ ]:
sc.pl.umap(nhood_adata_to_plot, color="CTHRC1", frameon=False)

In [ ]:
adata.obs.ann_level_2.value_counts()

Percentage of reference cells part of fibs:

In [ ]:
(
    adata.obs.loc[adata.obs.r_or_q == "r", "ann_level_2"] == "Fibroblast lineage"
).sum() / (adata.obs.r_or_q == "r").sum()

In [ ]:
adata.obs.ann_finest_level.nunique()

Percentage of cell types fibroblasts:

In [ ]:
(
    np.array(["fib" in str(ct) for ct in adata.obs.ann_finest_level.unique()])
).sum() / adata.obs.ann_finest_level.nunique()

check cell numbers per cell type and break down by study

In [ ]:
ct_n_per_study = pd.crosstab(
    adata.obs.loc[adata.obs.r_or_q == "r", "ann_level_2_merged"],
    adata.obs.loc[adata.obs.r_or_q == "r", "study"],
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ct_n_per_study.plot(kind="bar", stacked=True, ax=ax)
plt.grid(False)
plt.tight_layout()
plt.show()

### Macrophage sanity check:

In [ ]:
FIGURES[
    "grouped_barplot_mac_subtype_mapqc_scores"
] = mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_4_pred",
    group_category="r_or_q_ctr_or_case",
    box_groups_to_include=["Alveolar macrophages", "Interstitial macrophages"],
    figsize=(3, 5),
    palette=palette_1,
    min_cells_per_cat=50,
    pointsize=0.4,
    return_fig=True,
)
plt.xlabel("Cell type")
plt.show()

In [ ]:
adata_macs = adata[adata.obs.ann_level_3_merged == "Macrophages", :].copy()

In [ ]:
adata_macs.obs["ann_level_4_mapqc_score"] = analysis.clean_up_df_column(
    adata_macs.obs, column_name=obs_col, min_n_cells_per_cat=100
)

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby="ann_level_4_mapqc_score",
    groups=["Alveolar macrophages (q., mapqc>2)"],
    reference="Alveolar macrophages (ref.)",
    key_added="Alveolar macrophages (>2)",
)

In [ ]:
sc.tl.filter_rank_genes_groups(
    adata,
    key="Alveolar macrophages (>2)",
    key_added="Alveolar macrophages (>2) filtered",
)  # , max_out_group_fraction=0.1)
alv_mac_bad_map_deg_df = sc.get.rank_genes_groups_df(
    adata,
    group="Alveolar macrophages (q., mapqc>2)",
    key="Alveolar macrophages (>2) filtered",
).dropna(how="any")
alv_mac_bad_map_deg_df = alv_mac_bad_map_deg_df.loc[
    alv_mac_bad_map_deg_df.pvals_adj < 0.05, :
]  # only significant
alv_mac_bad_map_deg_df = alv_mac_bad_map_deg_df.loc[
    alv_mac_bad_map_deg_df.scores > 0, :
]  # only upregulated, for downregulated this filtering does not make sense

Get downregulated --> this gives too many results, not really useable without further filtering

In [ ]:
alv_mac_bad_map_down_deg_df = sc.get.rank_genes_groups_df(
    adata,
    group="Alveolar macrophages (q., mapqc>2)",
    key="Alveolar macrophages (>2)",
)
alv_mac_bad_map_down_deg_df = alv_mac_bad_map_down_deg_df.loc[
    alv_mac_bad_map_down_deg_df.pvals_adj < 0.05, :
]  # only significant
alv_mac_bad_map_down_deg_df = alv_mac_bad_map_down_deg_df.loc[
    alv_mac_bad_map_down_deg_df.scores < 0, :
]  # only downregulated

In [ ]:
sc.pl.dotplot(
    adata_macs,
    var_names=alv_mac_bad_map_deg_df.names.tolist() + ["FABP4"],
    groupby="ann_level_4_mapqc_score",
    standard_scale="var",
)

In [ ]:
# sc.pl.dotplot(
#     adata,
#     var_names={
#         "Down in IPF": ["FABP4", "PPARG"],
#         "Up in IPF": ["SPP1", "MERTK", "PLA2G7", "CCL2", "LGMN"],
#     },
#     groupby="ann_level_3_mapqc_score",
#     standard_scale="var",
# )

 LIPA expression steadily increasing in expression relatively early along the trajectory, ECM remodeling genes SPARC, GPC4, PALLD, CTSK, and MMP9 

In [ ]:
sc.pl.dotplot(
    adata_macs,
    var_names={
        "Down in IPF": ["FABP4", "PPARG"],
        "Up in IPF": [
            "SPP1",
            "MERTK",
            "PLA2G7",
            "CCL2",
            "IL1RN",
        ],
    },
    groupby="ann_level_4_mapqc_score",
    standard_scale="var",
)

### Output to store:

DEA tables:<br>
- SM mapqc>2 versus SM reference
- Pericyte mapqc>2 versus Pericyte reference
